# Gerar .txt de anotações vindas do GroundThuth

Autor:  Viviane da Rosa Sommer

Atualizado: 07/04/2025

## Objetivo: A partir da imagem original e da imagem de anotação, gerar o arquivo .txt para usar na YOLO.


## Importações Necessárias

In [ ]:
import glob
import numpy as np
import cv2
import os

## Declaração de Constantes e Modelos

In [ ]:
class_id = 0
CORAL_COLOR_HEX = "#2ca02c"
BACKGROUND_COLOR_HEX = "#ffffff"
CORAL_COLOR_BGR = tuple(int(CORAL_COLOR_HEX.lstrip('#')[i:i + 2], 16) for i in (4, 2, 0))
BACKGROUND_COLOR_BGR = tuple(int(BACKGROUND_COLOR_HEX.lstrip('#')[i:i + 2], 16) for i in (4, 2, 0))

## Funções Necessárias

In [ ]:
def close_contour(contour: np.ndarray) -> np.ndarray:
    """
    Ensures the contour is closed by adding the initial point at the end, if necessary.

    Args:
        contour (np.ndarray): The contour to be closed.

    Returns:
        np.ndarray: The closed contour.
    """
    if not np.array_equal(contour[0], contour[-1]):
        contour = np.vstack([contour, contour[0]])
    return contour

def normalize_coordinates(points: np.ndarray, image_width: int, image_height: int) -> np.ndarray:
    """
    Normalizes the coordinates of the points to the interval [0, 1].

    Args:
        points (np.ndarray): The points to be normalized.
        image_width (int): The width of the image.
        image_height (int): The height of the image.

    Returns:
        np.ndarray: The normalized points.
    """
    normalized_points = points / [image_width, image_height]
    return np.clip(normalized_points, 0, 1)

def calculate_bounding_box(contours: list[np.ndarray]) -> tuple[float, float, float, float]:
    """
    Calculates the normalized bounding box from a set of contours (outer and holes).

    Args:
        contours (list[np.ndarray]): The contours to calculate the bounding box from.

    Returns:
        tuple[float, float, float, float]: The x-center, y-center, width, and height of the bounding box.
    """
    all_points = np.vstack(contours)
    x_min, y_min = np.min(all_points, axis=0)
    x_max, y_max = np.max(all_points, axis=0)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return x_center, y_center, width, height

def create_color_mask(image: np.ndarray, target_color: tuple[int, int, int], threshold: int = 10) -> np.ndarray:
    """
    Creates a color mask for the given image and target color.

    Args:
        image (np.ndarray): The image to create the mask from.
        target_color (tuple[int, int, int]): The target color to create the mask for.
        threshold (int, optional): The threshold for the color range. Defaults to 10.

    Returns:
        np.ndarray: The color mask.
    """
    lower_bound = np.array(target_color) - threshold
    upper_bound = np.array(target_color) + threshold
    mask = cv2.inRange(image, lower_bound, upper_bound)
    return mask

## Gerar arquivos .txt

In [ ]:
for image_path in glob.glob("Datasets/Lote-4-5-Incompleto/label_images/**"):
    try:
        normalized_path = os.path.normpath(image_path)
        absolute_path = os.path.abspath(image_path)
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Could not read the image: {image_path}")

        image_height, image_width, _ = image.shape
        mask = create_color_mask(image, CORAL_COLOR_BGR)
        contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
        output_file = os.path.join("Datasets/Lote-4-5-Incompleto/yolo",
                                   os.path.basename(image_path).replace("label_", "")
                                   .replace(".jpg", ".txt").replace(".png", ".txt")
                                   .replace(".JPG", ".txt"))
        os.makedirs(os.path.dirname(output_file), exist_ok=True)

        try:
            with open(output_file, 'w') as file:
                for i, contour in enumerate(contours):
                    if hierarchy[0][i][3] == -1:
                        external_contour = close_contour(contour.reshape(-1, 2))
                        processed_contours = [external_contour]
                        hole_indices = [j for j in range(len(contours)) if hierarchy[0][j][3] == i]
                        for hole_index in hole_indices:
                            hole = close_contour(contours[hole_index].reshape(-1, 2))
                            processed_contours.append(hole)
                        normalized_contours = [
                            normalize_coordinates(c, image_width, image_height) for c in processed_contours
                        ]
                        x_center, y_center, width, height = calculate_bounding_box(normalized_contours)
                        segmentation_points = np.concatenate(normalized_contours).flatten()
                        line = f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f} " + \
                               " ".join(map(lambda x: f"{x:.6f}", segmentation_points)) + "\n"
                        file.write(line)
                print(f"File {output_file} generated successfully!")
        except IOError as e:
            print(f"Error writing to file {output_file}: {e}")
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
!jupyter nbconvert --to html Generate_YOLO_annot.ipynb